# Homework 3 — Waste per person: two files, three layers

**The question:** how much municipal waste does each of the 27 EU members generate per person, in 2024? You answer
it alone, in four steps. The first three are the things Session 3 taught; the fourth is the course's question,
*can I trust the answer?*

1. **Did every record arrive?** The World Bank's population pages (Block 5).
2. **Did every value survive?** Silver from Eurostat's waste file, with `CAST` (Block 6).
3. **Gold:** the two joined, waste per person (Block 6).
4. **Can you trust this answer?** A colleague's version of question 3, which runs with no error.

## The brief

> **Waste** is Eurostat's municipal waste generated (`wst_oper = 'GEN'`), in **thousand tonnes** (`unit = 'THS_T'`).
>
> **The members** are the 27 rows of the view `members`. Each has two codes: `eurostat_geo` in Eurostat's file
> (`EL` for Greece) and `worldbank_id` in the World Bank's (`GRC`).
>
> **Waste per person** is a member's total times 1,000,000 (thousand tonnes into kilograms), divided by its
> population in the same year: **kilograms per person**. Where a member has no total, it has no number: `NULL`. It
> need not equal Eurostat's own figure per person, because the two sources count population differently; you do not
> need to find out why.

## What you are given

The four setup cells below make these tables. **You never write them**; read them if you want to see how.

| View | One row is |
|---|---|
| `population` | one place or group and one year, from all four of the World Bank's pages: `iso3`, `country`, `year`, `population` |
| `population_metadata` | one page, and what it says: its number, how many pages, records per page, and the `total` the whole answer holds |
| `silver_split` | one cell of Eurostat's file: `geo`, `wst_oper`, `unit`, `year`; `raw_value` as Eurostat wrote it (`8295 p`, `:`), `number_text` the number still as text (`NULL` where Eurostat wrote `:`), `flag` the letter after it |
| `members` | one EU member: `name`, `eurostat_geo`, `worldbank_id` |

The setup makes a few other views on the way; you do not need them.

## How to answer

**Every query is yours, written in the empty cell under its step.** The text above each cell says what it must do;
the README's hints help if you are stuck. Under each question: your queries, **your count**, **the check**
(questions 2 to 4: one line, supplied; run it: each check says `PASS` or `FAIL`, and under a `FAIL` a line says what to
look at; a check tests only what it was written to test, so read your rows too), and **one sentence**. After a
restart, run the cells again from the top.

In [ ]:
import os
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 400)   # up to 400 rows in full; a longer result prints head and tail with "..." between

# Anchor to the project folder (DS1, Block 1), then stand there: every path below is from the project folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
print("Working in:", Path.cwd())   # must print this project's folder

con = duckdb.connect()             # an in-memory database; it reads the files in data/raw/ directly

In [ ]:
# Supplied: you never write this cell. It makes the layers you will look at, each a view named for its layer.
#
# bronze: the file exactly as it arrived, data/raw/env_wasmun.tsv. Every cell is read as text (all_varchar = true), so
# nothing is changed before we say so. One row per series; one column per year. The first column packs
# "freq,wst_oper,unit,geo" into one text, separated by commas: we call it `series`.
con.sql("""
    CREATE OR REPLACE VIEW bronze AS
    SELECT * FROM read_csv('data/raw/env_wasmun.tsv', delim = '\t', all_varchar = true, names = ['series'])
""")
# bronze_long: the same cells, nothing changed, one row per series, place and year. split_part cuts `series` at
# its commas (freq is A, annual, on every row, so it is left out); UNPIVOT turns each year's column into rows. `raw_value` is the cell exactly as written.
con.sql("""
    CREATE OR REPLACE VIEW bronze_long AS
    SELECT
           split_part(series, ',', 2) AS wst_oper,
           split_part(series, ',', 3) AS unit,
           split_part(series, ',', 4) AS geo,
           CAST(year AS INTEGER) AS year,
           raw_value
    FROM bronze
    UNPIVOT INCLUDE NULLS (raw_value FOR year IN (COLUMNS('^[0-9]{4}$')))
""")
# silver_split: silver's first step. A cell is a number, a space, and a flag if it has one ('628 e': 628,
# estimated), or ':' for not available. trim() takes the space off the end; split_part(…, ' ', 1) is the part before
# the space, split_part(…, ' ', 2) the part after it; NULLIF turns ':' into NULL, and an empty flag into NULL.
# number_text is still text: making it a number is your job, in question 2.
con.sql("""
    CREATE OR REPLACE VIEW silver_split AS
    SELECT wst_oper, unit, geo, year, raw_value,
           NULLIF(split_part(trim(raw_value), ' ', 1), ':') AS number_text,
           NULLIF(split_part(trim(raw_value), ' ', 2), '')  AS flag
    FROM bronze_long
""")

# Two lists of Eurostat's codes, as tables: each flag and what it means; each waste operation and what it is.
con.sql("""
    CREATE OR REPLACE VIEW flags AS
    SELECT * FROM read_csv('data/raw/codelist_obs_flag.tsv', delim = '\t', header = false, names = ['flag', 'meaning'])
""")
con.sql("""
    CREATE OR REPLACE VIEW waste_operations AS
    SELECT * FROM read_csv('data/raw/codelist_wst_oper.tsv', delim = '\t', header = false, names = ['wst_oper', 'meaning'])
""")
print("bronze:", con.sql("SELECT COUNT(*) FROM bronze").fetchone()[0], "rows;",
      "bronze_long:", con.sql("SELECT COUNT(*) FROM bronze_long").fetchone()[0], "rows;",
      "silver_split ready; flags and waste_operations ready")

# The 27 EU members, and each one's code in Eurostat's file and in the World Bank's.
con.sql("CREATE OR REPLACE VIEW members AS SELECT * FROM 'data/raw/country_codes.csv'")
print("members:", con.sql("SELECT COUNT(*) FROM members").fetchone()[0])

In [ ]:
# Supplied: the World Bank's population pages, made into tables for you (Block 5, Lab 5). You never write this; read it
# if you like. The API sent the series as JSON pages, saved in data/raw/api/worldbank/ exactly as they arrived.

def make_table(view, files, value, kind):
    """Turn World Bank pages into tables you can query. On each page DuckDB finds two things: the page's metadata
    (a record: an OBJECT) and the list of records (an ARRAY). For every file whose name matches `files` it makes:
        {view}_metadata   what each page says about itself: one row per page
        {view}_records    the records on the pages, one row each (unnest turns a list into rows)
        {view}            each record's fields as columns: ->> takes a value out by its name, CAST makes it a number
    A * in the file name matches any text: ..._page*.json is every page."""
    pages = f"read_json_auto('{files}')"
    con.sql(f"""
        CREATE OR REPLACE VIEW {view}_metadata AS
        SELECT CAST(json->>'page'     AS INTEGER) AS page,
               CAST(json->>'pages'    AS INTEGER) AS pages,
               CAST(json->>'per_page' AS INTEGER) AS per_page,
               CAST(json->>'total'    AS INTEGER) AS total
        FROM {pages}
        WHERE json_type(json) = 'OBJECT'
    """)
    con.sql(f"""
        CREATE OR REPLACE VIEW {view}_records AS
        SELECT unnest(CAST(json AS JSON[])) AS record
        FROM {pages}
        WHERE json_type(json) = 'ARRAY'
    """)
    con.sql(f"""
        CREATE OR REPLACE VIEW {view} AS
        SELECT record->>'countryiso3code'         AS iso3,
               record->'country'->>'value'        AS country,
               CAST(record->>'date'  AS INTEGER)  AS year,
               CAST(record->>'value' AS {kind})   AS {value}
        FROM {view}_records
    """)

make_table("population", "data/raw/api/worldbank/SP.POP.TOTL_page*.json", "population", "BIGINT")
print("views made: population, population_metadata")

In [ ]:
# The checks, written for you. Run this cell once, after the three above; you do not need to read it.
# Questions 2 and 3 each have a check cell that calls one of these.


def check(ok, claim, evidence=""):
    print("PASS" if ok else "FAIL", f"  {claim:<46} {evidence}")


def ready(view, columns):
    """True when the view exists, runs, and has its columns. If not, one FAIL line says what to do."""
    if not con.sql(f"SELECT 1 FROM duckdb_views() WHERE lower(view_name) = '{view}'").fetchall():
        check(False, f"no view named {view} yet", "write your query above, and run it")
        return False
    try:
        have = [r[0].lower() for r in con.sql(f"DESCRIBE {view}").fetchall()]
        con.sql(f"SELECT * FROM {view} LIMIT 1").fetchall()
    except duckdb.Error as e:
        check(False, f"your view {view} does not run")
        print(f"       {str(e).splitlines()[0]}")
        return False
    missing = [c for c in columns if c not in have]
    if missing:
        check(False, f"{view} has no column {missing[0]}")
        print(f"       name the column with AS {missing[0]} in your query")
        return False
    return True


def is_number(view, column):
    kind = {r[0].lower(): r[1] for r in con.sql(f"DESCRIBE {view}").fetchall()}[column]
    return kind in ("DOUBLE", "FLOAT", "BIGINT", "INTEGER", "HUGEINT") or kind.startswith("DECIMAL")


def shows_errors(f):
    def run():
        try:
            f()
        except duckdb.Error as e:
            check(False, "a check could not read your view")
            print(f"       {str(e).splitlines()[0]}")
            print("       read the columns of your view: are the numbers numbers?")
    return run


@shows_errors
def check_q2():
    """Question 2: your waste_silver, against silver_split."""
    if not ready('waste_silver', ['geo', 'year', 'thousand_tonnes']):
        return
    rows_in, numbers_in = con.sql("""
        SELECT COUNT(*), COUNT(number_text) FROM silver_split WHERE unit = 'THS_T' AND wst_oper = 'GEN'
    """).fetchone()
    rows, keys, numbers = con.sql("""
        SELECT COUNT(*), COUNT(DISTINCT (geo, year)), COUNT(thousand_tonnes) FROM waste_silver
    """).fetchone()
    # Row by row, on the rows your view has: is each number the one silver_split holds as text?
    off = con.sql("""
        SELECT COUNT(*) FROM waste_silver AS w
        JOIN silver_split AS s ON s.geo = w.geo AND s.year = w.year AND s.unit = 'THS_T' AND s.wst_oper = 'GEN'
        WHERE (w.thousand_tonnes IS NULL) <> (s.number_text IS NULL)
           OR abs(w.thousand_tonnes - CAST(s.number_text AS DOUBLE)) >= 0.005
    """).fetchone()[0]
    check(rows == keys == rows_in, "[1] one row per place and year", f"{rows:,} rows, {keys:,} distinct, {rows_in:,} in silver_split")
    if rows == rows_in and keys < rows:
        print("       a place and year twice: are geo and year taken from silver_split as they are?")
    elif rows < rows_in:
        print("       fewer rows than silver_split: does your WHERE keep only some places or years? It needs only the unit")
        print("       and the waste operation.")
    elif rows > rows_in:
        print("       more rows than silver_split: is the unit or the waste operation missing from your WHERE?")
    check(numbers == numbers_in and is_number("waste_silver", "thousand_tonnes"), "[2] every number survived, as a number",
          f"{numbers:,} of {numbers_in:,}")
    check(off == 0 and rows > 0, "[3] each number as the file has it", f"{off:,} rows differ" if rows else "no rows to compare")
    if off and rows == keys:
        print("       a number is not the file's: is it CAST(number_text AS DOUBLE), with the unit 'THS_T'?")


@shows_errors
def check_q3():
    """Question 3: your per_person_2024, against your waste_silver and the population."""
    if not ready('per_person_2024', ['geo', 'name', 'kg_per_person']):
        return
    n_members = con.sql("SELECT COUNT(*) FROM members").fetchone()[0]
    rows, keys, numbers, nulls = con.sql("""
        SELECT COUNT(*), COUNT(DISTINCT geo), COUNT(kg_per_person), COUNT(*) FILTER (WHERE kg_per_person IS NULL)
        FROM per_person_2024 WHERE geo IN (SELECT eurostat_geo FROM members)
    """).fetchone()
    all_rows = con.sql("SELECT COUNT(*) FROM per_person_2024").fetchone()[0]
    # The members whose 2024 total the file holds as a number.
    totals = con.sql("""
        SELECT COUNT(number_text) FROM silver_split
        WHERE unit = 'THS_T' AND wst_oper = 'GEN' AND year = 2024 AND geo IN (SELECT eurostat_geo FROM members)
    """).fetchone()[0]
    # Member by member: your 2024 total in thousand tonnes, times a million, over the 2024 population.
    off = con.sql("""
        SELECT COUNT(*) FROM per_person_2024 AS g
        JOIN members AS m ON m.eurostat_geo = g.geo
        JOIN waste_silver AS w ON w.geo = g.geo AND w.year = 2024
        JOIN population AS p ON p.iso3 = m.worldbank_id AND p.year = 2024
        WHERE (g.kg_per_person IS NULL) <> (w.thousand_tonnes IS NULL)
           OR abs(g.kg_per_person - w.thousand_tonnes * 1e6 / p.population) >= 0.005
    """).fetchone()[0]
    check(rows == keys == all_rows == n_members, "[1] one row per member", f"{all_rows} rows, {keys} members of {n_members}")
    if all_rows == 0:
        print("       no rows: does each join use that table's code? Greece above: EL is eurostat_geo, GRC is worldbank_id.")
    elif all_rows > n_members:
        print("       more than one row per member: does the WHERE keep 2024 only, for both tables?")
    elif all_rows < n_members:
        print("       fewer rows than members: does a filter drop the members with no total? They keep a row, with NULL.")
    check(numbers == totals and nulls == n_members - totals and is_number("per_person_2024", "kg_per_person"),
          "[2] a number where 2024 has a total", f"{numbers} numbers and {nulls} NULLs; {totals} members have a total")
    if all_rows == n_members and numbers == 0:
        print("       no numbers at all: does the population join use worldbank_id?")
    elif all_rows == n_members and numbers > totals:
        print("       more numbers than members with a 2024 total: is the year 2024 for both tables, and does a member")
        print("       with no total keep NULL?")
    check(off == 0 and all_rows > 0, "[3] total x 1,000,000 / population, for 2024",
          f"{off} rows differ by 0.005 kg or more" if all_rows else "no rows to compare")
    if off and all_rows == n_members and numbers == totals:
        print("       a number is off: thousand_tonnes * 1e6 / population, both for 2024.")
    # Your prediction, from the cell above question 3: printed beside your view's, never marked.
    said_rows, said_numbers = globals().get("my_rows"), globals().get("my_numbers")
    if said_rows is None or said_numbers is None:
        print("       your prediction: not written yet (my_rows and my_numbers, above question 3)")
    else:
        print(f"       your prediction: {said_rows} rows and {said_numbers} numbers; your view: {all_rows} rows and {numbers} numbers")


@shows_errors
def check_q4():
    """Question 4: your missing_from_report and per_person_repaired, against the files."""
    if not ready('missing_from_report', ['name']):
        return
    # The members the colleague's table has no row for.
    lacks = {r[0] for r in con.sql("""
        SELECT m.name FROM members AS m
        WHERE m.eurostat_geo NOT IN (SELECT geo FROM colleague_per_person)
    """).fetchall()}
    listed = [r[0] for r in con.sql("SELECT name FROM missing_from_report").fetchall()]
    check(len(listed) == len(lacks) and set(listed) == lacks, "[1] the members the report has no row for",
          f"{len(listed)} listed; the report lacks {len(lacks)}")
    if len(listed) == 0:
        print("       no rows: is it a LEFT JOIN from members, keeping the rows that found no partner (IS NULL)?")
    elif len(listed) > len(lacks):
        print("       too many rows: does your WHERE keep only the members that found no partner (IS NULL)?")
    if not ready("per_person_repaired", ["geo", "name", "kg_per_person"]):
        return
    n_members = con.sql("SELECT COUNT(*) FROM members").fetchone()[0]
    rows, keys, numbers = con.sql(
        "SELECT COUNT(*), COUNT(DISTINCT geo), COUNT(kg_per_person) FROM per_person_repaired").fetchone()
    # Member by member, from the file and the population: the 2024 total x 1,000,000 over the 2024 population.
    off = con.sql("""
        SELECT COUNT(*) FROM members AS m
        JOIN silver_split AS s ON s.geo = m.eurostat_geo AND s.unit = 'THS_T' AND s.wst_oper = 'GEN' AND s.year = 2024
        JOIN population AS p ON p.iso3 = m.worldbank_id AND p.year = 2024
        LEFT JOIN per_person_repaired AS r ON r.geo = m.eurostat_geo
        WHERE (r.kg_per_person IS NULL) <> (s.number_text IS NULL)
           OR abs(r.kg_per_person - CAST(s.number_text AS DOUBLE) * 1e6 / p.population) >= 0.005
    """).fetchone()[0]
    check(rows == keys == n_members, "[2] the repair: one row per member", f"{rows} rows, {keys} members of {n_members}")
    if rows < n_members:
        print("       still short of the members: which code of members does the population join use?")
    check(off == 0 and rows > 0, "[3] the repair: each member's number", f"{numbers} numbers, {off} members differ")
    if off and rows == n_members:
        print("       a member has no number where it has a total: does the population join use the World Bank's code?")


print("checks ready: check_q2, check_q3, check_q4")

---

# Your work starts here

## 1. Did every record arrive?

*Block 5, cells 6 and 7: what the pages say they hold, and how many rows the table holds.*

The World Bank sent the population in four pages. The cell below shows what each page says about itself, and the
first rows of `population`.

In [ ]:
# Q1, supplied: what each page says about itself, then the first rows of the table made from all four.
display(con.sql("SELECT * FROM population_metadata ORDER BY page").df())
con.sql("SELECT * FROM population LIMIT 5").df()

**(a) Your count:** the rows of `population`, beside the `total` the pages state (it is in
`population_metadata`; every page states the same total, so take page 1's).

In [ ]:
# Q1(a), your count. The rows of population, beside the total the pages state.
# A cell shows only its last result: wrap the others in display(...), or use a new cell for each.

**(b) The members' records.** Gold will need each member's population in 2024. Count the rows you get when
you join `members` to `population` on the World Bank's code (`worldbank_id` in `members`, `iso3` in `population`),
keeping 2024 only. If every member's record is there, it is 27.

In [ ]:
# Q1(b), your count. The members joined to population on the World Bank's code, 2024 only.

**Sentence.** _(did every record arrive? say how your counts show it)_

## 2. Did every value survive? Silver from the waste file

*Block 6, cell 2 (silver: the value made a number with `CAST`) and cell 6 (bronze against silver: rows and
numbers).*

Why `CAST` and not `TRY_CAST`: `CAST` stops with an error on a value it cannot read, and names it; `TRY_CAST` turns it
into `NULL` without a word. The cell below shows the columns you will use.

In [ ]:
# Q2, supplied: four members' totals, 2024 only, to show the columns. Which has no number_text, and why?
# (Your view keeps every place and every year: this look's geo and year filters are not part of it.)
con.sql("""
    SELECT geo, year, raw_value, number_text, flag FROM silver_split
    WHERE unit = 'THS_T' AND wst_oper = 'GEN' AND year = 2024 AND geo IN ('HR', 'BE', 'DE', 'AT')
    ORDER BY geo
""").df()

**(a) Your query: the view `waste_silver`.** From `silver_split`, the rows of the brief's waste (its unit and
its waste operation; every place and every year), with three columns: `geo`, `year`, and **`thousand_tonnes`**, which
is `number_text` made a number with `CAST`. Then show a few of its rows.

In [ ]:
# Q2(a), your query. The view waste_silver, then a few of its rows.
# A cell shows only its last result: wrap the others in display(...), or use a new cell for each.

**(b) Your count:** the numbers before your step and after it, on the same rows. Before: how many rows of
`silver_split`, with your view's two filters and no others, have a `number_text`. After: how many rows of
`waste_silver` have a `thousand_tonnes` (your view already holds only those rows, so no `WHERE`). `COUNT(column)`
counts the rows where that column is not `NULL`. If no value was lost, the two are equal.

In [ ]:
# Q2(b), your count. The numbers before your step, and after it.
# A cell shows only its last result: wrap the others in display(...), or use a new cell for each.

In [ ]:
# Q2 check, supplied. It checks your view; it does not read your count. Your count's numbers should be the
# ones on line [2]. If they are not, check that both sides of your count cover the same rows and count the
# same thing (the numbers: COUNT of the number column, on both sides).
check_q2()

**(c) The numbers, year by year.** One row per year of `waste_silver`: the year, its rows, and its numbers
(`COUNT(*)` and `COUNT(thousand_tonnes)`), with the fewest numbers first. Which year has the fewest?

In [ ]:
# Q2(c), your query. One row per year: the year, its rows, its numbers; the fewest numbers first.

**(d) The members with no number.** Which members have no number for 2024 in `waste_silver`? Join
`members` to `waste_silver` on Eurostat's code (`eurostat_geo` in `members`, `geo` in `waste_silver`), keep 2024 and
the rows where `thousand_tonnes` is `NULL`, and show their names.

In [ ]:
# Q2(d), your query. The names of the members with no number for 2024 in waste_silver.

**Sentence.** _(did every value survive your step? say how your count shows it, and which year has the fewest numbers)_

## Your prediction, before question 3

Question 3's gold starts from the 27 members and joins `waste_silver` and `population`, for 2024. **Before you write
it:** how many rows will it have, and how many of them will have a number? Write both in the cell below, with one
reason. Question 3's check prints them beside your view's. A wrong prediction costs nothing; the reason is what
counts.

In [ ]:
# Your prediction, before you write question 3: two numbers, and one reason.
my_rows = None        # how many rows per_person_2024 will have
my_numbers = None     # how many of them will have a kg_per_person
my_reason = ""        # because ...

## 3. Gold: waste per person, 2024

*Block 6, cell 5 (gold: two silvers joined) and cell 3 (`COUNT(*)` and `COUNT` of a column).*

**Your query: the view `per_person_2024`**, one row per member, with the columns `geo`, `name` and **`kg_per_person`**
(the brief's waste per person). Start from `members`, join your `waste_silver` on Eurostat's code (as in 2(d)) and
`population` on the World Bank's (as in 1(b)), and keep 2024 for both. Then show it, the name and `kg_per_person`
rounded to one decimal, highest first.

A sense of size: every member with a total is between about 350 and 800 kg per person; a number like 0.0007 or
700,000 means a factor or a division went the wrong way. The cell below shows Greece in all three tables, so you can
see which codes match.

In [ ]:
# Q3, supplied: Greece in members, in your waste_silver (2024), and in population (2024). Which codes match?
try:
    display(con.sql("SELECT * FROM members WHERE name = 'Greece'").df())
    display(con.sql("SELECT * FROM waste_silver WHERE geo = 'EL' AND year = 2024").df())
    display(con.sql("SELECT * FROM population WHERE iso3 = 'GRC' AND year = 2024").df())
except duckdb.Error as e:
    print("Not yet: make waste_silver in question 2 first.")
    print("       ", str(e).splitlines()[0])

In [ ]:
# Q3, your query. The view per_person_2024, then shown, highest first.
# A cell shows only its last result: wrap the others in display(...), or use a new cell for each.

**Your count:** the rows of `per_person_2024`, and how many of them have a number (`COUNT(*)`, and `COUNT` of
`kg_per_person`).

In [ ]:
# Q3, your count. The rows of per_person_2024, beside the rows with a number.

In [ ]:
# Q3 check, supplied. It checks your view; it does not read your count. Your count's numbers should be the
# ones on lines [1] and [2]. If they are not, check that both sides of your count cover the same rows and count the
# same thing (COUNT(*) for the rows, COUNT(kg_per_person) for the numbers).
check_q3()

**Sentence.** _(your prediction against your count, and why the rows are what they are; then the highest and the lowest member with a number)_

## 4. Can you trust this answer? A colleague's report

*Block 6, cell 5 (a join on the name and year); Block 3, cell 9 (the rows with no partner: `LEFT JOIN` and
`IS NULL`).*

A colleague answered question 3 on the same files and sent this report:

> **Waste per person, 2024.** Denmark is highest, at 754.7 kg per person, and Estonia lowest, at 374.5. **19 of the 27
> members** have a figure for 2024.

Their query is in the cell below, and it runs with no error. They joined the population on the member's **name**, as
Block 6's cell 5 joined GDP to population.

In [ ]:
# Q4, supplied: the colleague's query, as they wrote it, and their table, highest first.
con.sql("""
    CREATE OR REPLACE VIEW colleague_per_person AS
    SELECT m.eurostat_geo AS geo, m.name,
           CAST(s.number_text AS DOUBLE) * 1e6 / p.population AS kg_per_person
    FROM members AS m
    JOIN silver_split AS s ON s.geo = m.eurostat_geo
    JOIN population   AS p ON p.country = m.name
    WHERE s.unit = 'THS_T' AND s.wst_oper = 'GEN' AND s.year = 2024 AND p.year = 2024
""")
con.sql("""
    SELECT name, ROUND(kg_per_person, 1) AS kg_per_person_1dp
    FROM colleague_per_person
    ORDER BY kg_per_person DESC
""").df()

**(a) Check the report before you trust it.** It says 19 of the 27 members have a figure. Which count
on `colleague_per_person` would test that claim? Say it in the comment, then write it, and put it beside your count in
question 3. Which number in the report is wrong?

In [ ]:
# Q4(a), your count. The count that tests the report's claim:

**(b) Which member?** Write a view named **`missing_from_report`**, with one column, `name`: the members that
have no row in `colleague_per_person` (a `LEFT JOIN` from `members`, and the rows that found no partner). Then show
it.

In [ ]:
# Q4(b), your query. The view missing_from_report, then its rows.
# A cell shows only its last result: wrap the others in display(...), or use a new cell for each.

**(c) Why?** Look at that member's row in `members`, and at its row in `population` for 2024 (find it by
its `worldbank_id`). Compare the names.

In [ ]:
# Q4(c), your query. That member's row in members, and its 2024 row in population.
# A cell shows only its last result: wrap the others in display(...), or use a new cell for each.

**(d) The repair.** The cell below is the colleague's query, copied, saved as `per_person_repaired`. It
runs as it is, and is still a member short. Change their join so that every member keeps its row, and finish the
comment on its first line: why did the report lose a member?

In [ ]:
# Q4(d), the repair: the colleague's query, copied. Change it so every member keeps its row, and finish the comment.
con.sql("""
    -- Why the report lost a member: ________
    CREATE OR REPLACE VIEW per_person_repaired AS
    SELECT m.eurostat_geo AS geo, m.name,
           CAST(s.number_text AS DOUBLE) * 1e6 / p.population AS kg_per_person
    FROM members AS m
    JOIN silver_split AS s ON s.geo = m.eurostat_geo
    JOIN population   AS p ON p.country = m.name
    WHERE s.unit = 'THS_T' AND s.wst_oper = 'GEN' AND s.year = 2024 AND p.year = 2024
""")
con.sql("SELECT COUNT(*) AS members, COUNT(kg_per_person) AS with_a_number FROM per_person_repaired").df()

In [ ]:
# Q4 check, supplied. It checks your two views, missing_from_report and per_person_repaired; it does not
# read your count.
check_q4()

**Sentence.** _(which number in the report is wrong, how your count shows it, and why it was wrong)_

## The end of the course: a short reflection

Three lines, a sentence or two each, about **your own** work in this course: specifics, not general advice.

1. Which kind of silent failure was hardest for you to see: what one row is, a `NULL`, a join that left rows out or
   repeated them, a document read only in part, or a cast that swallowed a value? Name the lab or homework.
2. A count you wrote in this homework: what mistake would it have caught?
3. One habit from this course you will keep in your next job or project.

_(your three lines here)_

## AI use

One line: what you asked an assistant, and which cell checked its answer. *Did not use AI* is a fine answer.

_(your line here)_

---

**Before you upload:** *Restart*, then *Run All*, then save. Every cell must run without an error: leave an unfinished
query as comments (put `#` in front of each line). Question 3 needs question 2's `waste_silver`; if question 2 will
not work, post on the Moodle forum and we will help you get past it. Upload `notebooks/report.ipynb` to the Homework 3
slot on Moodle. That is the whole submission: no zip, no other file.